# 09 — Measurement semantics audit

**CPU runtime → Runtime → Run all → upload your completed Notebook 08 results ZIP.**

This checks the origin and meaning of the zero packet-count example, exercises the original feature extractor, and freezes 128 additional malicious development inputs. It does not run new attacks or train/score an IDS. The existing Notebook 08 results and validity profile remain unchanged.

A fresh run downloads approximately **472 MB** of CSV data. A GPU is unnecessary. Processing is lightweight; download time depends on the connection.


## 1. Create a new session

Each normal Colab execution clones into a new folder. Earlier checkouts and results are preserved. Leave `INPUT_ZIP` empty to use the upload button.


In [ ]:
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

INPUT_ZIP = "" #@param {type:"string"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}
INPUT_ZIP = os.environ.get("PHD_INPUT_ZIP", INPUT_ZIP)
if os.environ.get("PHD_NO_DOWNLOAD") == "1":
    DOWNLOAD_RESULTS = False

def unique_name(prefix):
    return prefix + "_" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

REPO_URL = "https://github.com/Abdulstar/MyPhDThesis.git"
if os.environ.get("PHD_WORKSPACE"):
    ROOT = Path(os.environ["PHD_WORKSPACE"]).expanduser().resolve()
    if not (ROOT / "baseline/measurement_semantics.py").is_file():
        raise RuntimeError("PHD_WORKSPACE must contain the new Notebook 09 source files.")
else:
    ROOT = Path("/content") / unique_name("MyPhDThesis_Semantics")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
HOST_PYTHON, HOST_VERSION = sys.executable, sys.version
if not ((3, 11) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError("The audit requirements support Python 3.11–3.13. Use a compatible Colab runtime.")
if (ROOT / ".git").exists():
    COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
    STATUS = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).splitlines()
else:
    COMMIT, STATUS = "local_source_snapshot_no_git", ["Local validation directory"]
SESSION = ROOT / "semantics_runs" / unique_name("measurement_semantics")
SESSION.mkdir(parents=True, exist_ok=False)
SOURCES = [
    "baseline/measurement_semantics.py", "baseline/measurement_semantics_protocol.json",
    "baseline/requirements-semantics.txt", "scripts/review_notebook08_colab.py",
    "tests/test_measurement_semantics.py", "notebooks/09_Measurement_Semantics_Audit.ipynb",
    "research_sources/ctu13_2f1624c/aggregated_features_bro_logs.py",
    "research_sources/ctu13_2f1624c/LICENSE", "research_sources/ctu13_2f1624c/PROVENANCE.md",
    "docs/MEASUREMENT_SEMANTICS_GUIDE.md", "docs/MEASUREMENT_SEMANTICS_FINDINGS_2026-10-21.md",
]
source_hashes = {}
for name in SOURCES:
    original = ROOT / name
    target = SESSION / "source_snapshot" / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(original, target)
    source_hashes[name] = hashlib.sha256(original.read_bytes()).hexdigest()
context = dict(project_repository=REPO_URL, project_commit=COMMIT, project_status=STATUS,
               notebook_python=HOST_VERSION, notebook_executable=HOST_PYTHON,
               source_sha256=source_hashes, experiment="measurement_semantics_audit",
               session_utc=dt.datetime.now(dt.timezone.utc).isoformat())
(SESSION / "session_context.json").write_text(json.dumps(context, indent=2) + "\n")

def run_command(command, log_name):
    environment = dict(os.environ, PYTHONUNBUFFERED="1")
    with (SESSION / log_name).open("x") as log:
        process = subprocess.Popen([str(x) for x in command], cwd=ROOT, env=environment,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        result = process.wait()
    if result:
        raise RuntimeError(f"Command failed ({result}); see {log_name}. Export the partial session using the last cell.")

print("New session:", SESSION)
print("Source revision:", COMMIT)


In [ ]:
if INPUT_ZIP.strip():
    INPUT = Path(INPUT_ZIP).expanduser().resolve()
else:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Set INPUT_ZIP to your completed Notebook 08 results ZIP.")
    upload_dir = SESSION / "upload"
    upload_dir.mkdir()
    previous_cwd = Path.cwd()
    try:
        os.chdir(upload_dir)
        uploaded = files.upload()
    finally:
        os.chdir(previous_cwd)
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one completed Notebook 08 ZIP.")
    filename = next(iter(uploaded))
    if Path(filename).name != filename:
        raise ValueError("Unexpected upload filename")
    INPUT = upload_dir / filename
    del uploaded
if not INPUT.is_file() or INPUT.suffix.lower() != ".zip":
    raise ValueError("INPUT_ZIP must point to a ZIP file.")
print("Input:", INPUT.name)


## 2. Prepare the lightweight audit environment

This isolated environment contains NumPy and pandas, without model dependencies. The notebook kernel is unchanged. Six focused tests check the measurement counterexamples using the original extractor.


In [ ]:
ENV = ROOT / ".semantics-env"
if not (ENV / "bin/python").is_file():
    run_command([HOST_PYTHON, "-m", "venv", ENV], "create_environment.log")
PYTHON = ENV / "bin/python"
run_command([PYTHON, "-m", "pip", "install", "-r", ROOT / "baseline/requirements-semantics.txt"], "install.log")
run_command([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-p", "test_measurement_semantics.py", "-v"], "tests.log")
run_command([PYTHON, "-m", "pip", "freeze"], "environment_packages.txt")
assert sys.executable == HOST_PYTHON and sys.version == HOST_VERSION


## 3. Run the source and clean-control audit

The workflow verifies both data-file hashes, compares the authors' Scenario 1 aggregates with the benchmark, and audits only the 143,046-row train/development prefix in two precisions. It preserves the row-1503 counterexample.

The follow-up input sample excludes the 32 pilot rows. The protocol keeps the existing validity rules and freezes a feature-space comparison; no packet-duration rule is adopted and no new attack is executed.


In [ ]:
RUN = SESSION / "audit"
command = [PYTHON, ROOT / "baseline/measurement_semantics.py", "--input-zip", INPUT,
           "--out", RUN, "--assets", ROOT / "assets"]
for flag, variable in [("--dataset", "PHD_DATASET"), ("--scenario1", "PHD_SCENARIO1")]:
    if os.environ.get(variable):
        command += [flag, Path(os.environ[variable]).expanduser().resolve()]
run_command(command, "audit.log")
print("Completed audit:", RUN)


## 4. Read the outcome

`packet_duration_rule_adopted: false` is intentional. The audit explains why neither zero nor positive packet totals identify measurement completeness. A successful audit is not improved IDS accuracy.

The 128 new development records are saved for the next attack comparison. They have not been attacked in this notebook. Development data is not an independent held-out test, and the inherited all-row scaler limitation remains.


In [ ]:
report = json.loads((RUN / "report.json").read_text())
assert report["status"] == "completed"
assert report["test_rows_evaluated"] == 0 and not report["model_scoring_performed"]
print("Source comparison:", json.dumps(report["source_bridge"], indent=2))
print("Synthetic vector matches row 1503:", report["synthetic_row1503_match"])
print("Packet-duration rule adopted:", report["packet_duration_rule_adopted"])
print("New development inputs:", report["followup"]["sample_size"])
print("New attacks executed:", report["followup"]["attacks_executed"])
print("Audit seconds (including asset downloads, excluding installation):", round(report["wall_seconds"], 2))
try:
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(RUN / "conditional_bound_flags.csv"))
except ImportError:
    print((RUN / "conditional_bound_flags.csv").read_text())


## 5. Download and share the new ZIP

Share this new results ZIP for review. It contains the source snapshot, controlled fixtures, control results, provenance evidence and the frozen next sample. Large dataset files and the uploaded 08 archive are excluded; their hashes are recorded.

If a previous cell failed, this cell can still export the partial session. A partial export is not a completed audit. Previous exports are never overwritten.


In [ ]:
import zipfile
EXPORTS = ROOT / "exports"
EXPORTS.mkdir(exist_ok=True)
archive = EXPORTS / (unique_name(SESSION.name + "_results") + ".zip")
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(SESSION.rglob("*")):
        if path.is_file() and "upload" not in path.relative_to(SESSION).parts:
            z.write(path, path.relative_to(SESSION.parent))
print("Results ZIP:", archive)
print("SHA256:", hashlib.sha256(archive.read_bytes()).hexdigest())
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive))
